# visual-tcav — End-to-End Demo

This notebook demonstrates the full capabilities of the `visual-tcav` package
using ResNet50 pretrained on ImageNet and concept images from the
[DTD (Describable Textures Dataset)](https://www.robots.ox.ac.uk/~vgg/data/dtd/).

**What you will see:**
- How to inspect available layers before configuring the explainer
- Two ways to load a model (string or nn.Module)
- How `explain()` works without an explicit `predict()` call
- Concept maps across multiple CNN layers
- How to plug in a custom CAV computation function
- Caching behavior and how to clear the cache
- Global attribution statistics with confidence intervals
- How to interpret concept maps and attribution scores

**Reference paper:**
De Santis et al., *Visual-TCAV: Concept-based Attribution and Saliency Maps
for Post-hoc Explainability in Image Classification*, 2025.
[arXiv:2411.05698](https://arxiv.org/abs/2411.05698)

---

## 0. Installation

```bash
pip install visual-tcav
```

For the Text-to-Concept extension (requires CLIP):
```bash
pip install visual-tcav[text-to-concept]
```

## 1. Setup

All paths are fully configurable — set them to wherever your files live.
There is no forced folder structure.

In [ ]:
import os
import torch
import torchvision.models as models
import matplotlib.pyplot as plt

from visual_tcav import (
    available_layers,
    LocalVisualTCAV,
    GlobalVisualTCAV,
    Cav,
)

print(f"PyTorch version: {torch.__version__}")
print(f"Device: {'GPU' if torch.cuda.is_available() else 'CPU'}")

# All paths are individually configurable
CONCEPT_DIR     = "./data/concept_images"
RANDOM_DIR      = "./data/concept_images/random"
TEST_IMAGES_DIR = "./data/test_images"
CACHE_DIR       = "./data/.cache"

for name, path in [
    ("concept_images", CONCEPT_DIR),
    ("test_images",    TEST_IMAGES_DIR),
    ("random",         RANDOM_DIR),
]:
    status = "OK" if os.path.exists(path) else "NOT FOUND"
    print(f"  [{status}] {name}: {path}")

## 2. Inspect available layers

Use `available_layers()` **before** instantiating the explainer to decide
which layers to analyze. Deeper layers capture higher-level concepts.

In [ ]:
# From a model name string
available_layers("resnet50")

In [ ]:
# Also works with an nn.Module directly
my_resnet = models.resnet50(weights=models.ResNet50_Weights.DEFAULT)
available_layers(my_resnet, model_name="resnet50")

---

## 3. LocalVisualTCAV — model string

The simplest usage. The package loads the model with default ImageNet weights.

Supported: `resnet18`, `resnet50`, `resnet101`, `vgg16`, `vgg19`.

In [ ]:
tcav_local = LocalVisualTCAV(
    model="resnet50",
    test_image_path=os.path.join(TEST_IMAGES_DIR, "zebra.jpg"),
    concept_names=["striped", "dotted", "zigzagged"],
    concept_base_dir=CONCEPT_DIR,
    random_dir=RANDOM_DIR,
    layer_names=["layer4"],
    n_classes=3,
    cache_dir=CACHE_DIR,
)

### 3.1 Predict (optional)

`predict()` is **optional** — you never need to call it explicitly.
`explain()` will call it automatically if you skip it.

Call it directly only if you want to see the predictions before running the explanation.

In [ ]:
# Optional — call predict() explicitly to inspect predictions first
predictions = tcav_local.predict()
predictions.info(num_of_classes=5)

### 3.2 Explain

`explain()` runs the full Visual-TCAV pipeline.

**Caching:** by default (`force_recompute=False`), CAVs and random activations
are cached to `cache_dir` and reloaded on subsequent runs. This avoids
redundant computation when experimenting with the same model and concepts.

Use `force_recompute=True` to ignore the cache and recompute everything fresh.

In [ ]:
# Default: use cache if available (fast on subsequent runs)
tcav_local.explain()

### 3.3 Visualize

In [ ]:
tcav_local.plot(figsize=(14, 10))

---

## 4. explain() without predict() — auto-call behavior

`explain()` automatically calls `predict()` internally if you have not done so.
This means you can go straight from construction to explanation.

In [ ]:
tcav_no_predict = LocalVisualTCAV(
    model="resnet50",
    test_image_path=os.path.join(TEST_IMAGES_DIR, "honeycomb.jpg"),
    concept_names=["honeycombed", "waffled", "chequered"],
    concept_base_dir=CONCEPT_DIR,
    random_dir=RANDOM_DIR,
    layer_names=["layer4"],
    cache_dir=CACHE_DIR,
)

# No predict() call — explain() handles it automatically
tcav_no_predict.explain()
tcav_no_predict.plot(figsize=(12, 5))

---

## 5. Cache management

### How caching works

Visual-TCAV caches two types of results to disk:
- **Random activations** — pooled CNN activations of random images at each layer.
  These only depend on the model and layer, not on the test image or concepts.
- **CAVs** — the Concept Activation Vectors for each (concept, layer) pair.
  These depend on concept images, model, and layer — not on the test image.

Both are saved as `.joblib` files in `cache_dir` and reloaded automatically
on subsequent calls to `explain()`. This avoids redundant computation when
experimenting with different test images but the same model and concepts.

### When to use `force_recompute=True`

Use it when:
- You have changed concept images and want fresh CAVs
- You have changed the model and want fresh activations
- You suspect the cache is stale or corrupted

In [ ]:
# Force recomputation — ignores any existing cached files
tcav_local.explain(force_recompute=True)

### Clearing the cache

Use `clear_cache()` to delete all cached files at once.

In [ ]:
# Delete all cached files
tcav_local.clear_cache()

# After clearing, explain() recomputes everything from scratch
tcav_local.explain()

### Disabling caching entirely

Set `cache_dir=None` to disable caching completely.
Every call to `explain()` will recompute from scratch.

In [ ]:
tcav_no_cache = LocalVisualTCAV(
    model="resnet50",
    test_image_path=os.path.join(TEST_IMAGES_DIR, "zebra.jpg"),
    concept_names=["striped"],
    concept_base_dir=CONCEPT_DIR,
    random_dir=RANDOM_DIR,
    layer_names=["layer4"],
    cache_dir=None,  # caching disabled
)
tcav_no_cache.explain()

---

## 6. LocalVisualTCAV — nn.Module

Pass any PyTorch model directly. Use this for fine-tuned models or
non-standard weights. Set `model_name` to enable auto-loading of labels.

In [ ]:
tcav_module = LocalVisualTCAV(
    model=my_resnet,
    model_name="resnet50",
    test_image_path=os.path.join(TEST_IMAGES_DIR, "zebra.jpg"),
    concept_names=["striped", "dotted"],
    concept_base_dir=CONCEPT_DIR,
    random_dir=RANDOM_DIR,
    layer_names=["layer3", "layer4"],
    cache_dir=CACHE_DIR,
)
tcav_module.explain()
tcav_module.plot(figsize=(16, 5))

---

## 7. Concept maps across multiple layers

Analyzing the same concept at different layers shows how detection evolves
with depth. Early layers detect simple textures; later layers detect patterns.

In [ ]:
tcav_multilayer = LocalVisualTCAV(
    model="resnet50",
    test_image_path=os.path.join(TEST_IMAGES_DIR, "zebra.jpg"),
    concept_names=["striped"],
    concept_base_dir=CONCEPT_DIR,
    random_dir=RANDOM_DIR,
    layer_names=["layer2", "layer3", "layer4"],
    cache_dir=CACHE_DIR,
)
tcav_multilayer.explain()
tcav_multilayer.plot(figsize=(16, 5))

---

## 8. Custom CAV computation function

Replace the default centroid difference method with any custom function
via the `cav_fn` parameter (Strategy Pattern).

The function must accept two `[N, C]` tensors and return a `Cav` object.

In [ ]:
def normalized_centroid_cav(
    concept_features: torch.Tensor,
    random_features: torch.Tensor,
) -> Cav:
    """
    Custom CAV: centroid difference normalized to unit length.

    Normalizing ensures attribution scores are comparable across
    concepts with different activation magnitudes.
    """
    direction = (
        torch.mean(concept_features, dim=0)
        - torch.mean(random_features, dim=0)
    )
    direction = direction / (torch.norm(direction) + 1e-10)
    return Cav(direction=direction)


tcav_custom = LocalVisualTCAV(
    model="resnet50",
    test_image_path=os.path.join(TEST_IMAGES_DIR, "zebra.jpg"),
    concept_names=["striped", "dotted"],
    concept_base_dir=CONCEPT_DIR,
    random_dir=RANDOM_DIR,
    layer_names=["layer4"],
    cache_dir=CACHE_DIR,
    cav_fn=normalized_centroid_cav,
)

# force_recompute=True because the CAV function changed
tcav_custom.explain(force_recompute=True)
tcav_custom.plot(figsize=(12, 5))

### Default vs custom CAV — comparison

In [ ]:
import numpy as np

concepts = ["striped", "dotted"]
layer = "layer4"
top_class = tcav_local.target_classes[0]

default_scores = [
    tcav_local.computations[layer][c].attributions.get(
        top_class, torch.tensor(0.0)
    ).item()
    for c in concepts
]
custom_scores = [
    tcav_custom.computations[layer][c].attributions.get(
        top_class, torch.tensor(0.0)
    ).item()
    for c in concepts
]

x = np.arange(len(concepts))
width = 0.35

fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(x - width/2, default_scores, width,
       label="Default (centroid diff)", color="steelblue")
ax.bar(x + width/2, custom_scores, width,
       label="Custom (normalized)", color="coral")
ax.set_xticks(x)
ax.set_xticklabels(concepts)
ax.set_ylabel("Attribution score")
ax.set_title(
    f"Default vs Custom CAV — layer4 — "
    f"{tcav_local.model_wrapper.id_to_label(top_class)}"
)
ax.legend()
ax.set_ylim(bottom=0)
plt.tight_layout()
plt.show()

---

## 9. GlobalVisualTCAV

Answers: **Does this concept consistently influence predictions for this class?**

In [ ]:
tcav_global = GlobalVisualTCAV(
    model="resnet50",
    test_images_dir=os.path.join(TEST_IMAGES_DIR, "zebra"),
    concept_names=["striped", "dotted", "zigzagged"],
    concept_base_dir=CONCEPT_DIR,
    random_dir=RANDOM_DIR,
    layer_names=["layer4"],
    n_classes=3,
    max_test_images=50,
    cache_dir=CACHE_DIR,
)

tcav_global.explain()

In [ ]:
tcav_global.statsInfo()

In [ ]:
tcav_global.plot(figsize=(10, 5))

---

## 10. Saving results to disk

In [ ]:
os.makedirs("./results", exist_ok=True)

tcav_local.plot(
    figsize=(14, 10),
    save_path="./results/zebra_local_explanation.png",
)
tcav_global.plot(
    figsize=(10, 5),
    save_path="./results/zebra_global_explanation.png",
)

print("Figures saved to ./results/")

---

## 11. Interpreting the results

### Concept map
- **Red/yellow** — the CNN strongly detects the concept here
- **Black** — the concept is absent
- Upscaled from layer resolution (7×7 for ResNet50 layer4) to 224×224
  using bilinear interpolation for a smooth overlay

### Attribution score
- Scores are **not percentages** — they are raw scalar values
- What matters is the **relative ranking** between concepts
- A much higher score for `striped` than `dotted` on a zebra image means
  the model uses stripes more than dots when classifying zebras

### Global confidence interval
| CI | Interpretation |
|---|---|
| Narrow, e.g. [0.18, 0.22] | Concept consistently matters ✅ |
| Wide, e.g. [0.02, 0.38] | Concept matters for some images but not others ⚠️ |
| Includes 0 | Concept does not reliably influence predictions ❌ |

---

## 12. Text-to-Concept (optional — requires CLIP)

Generates CAVs from plain text instead of concept images.

```bash
pip install visual-tcav[text-to-concept]
```

In [ ]:
try:
    from visual_tcav import TextToConcept
    t2c = TextToConcept(model_wrapper=tcav_local.model_wrapper)
    t2c.load_aligner("./aligners/resnet50_layer4.pt")
    cav_from_text = t2c.get_cav_from_text("stripes", layer_name="layer4")
    print(f"CAV from text: direction shape = {cav_from_text.direction.shape}")
except ImportError:
    print("CLIP not installed. Run: pip install visual-tcav[text-to-concept]")
except FileNotFoundError:
    print("Linear Aligner not found. Train it first with train_aligners.py")

---

## Summary

| Feature | How to use |
|---|---|
| **Inspect layers** | `available_layers("resnet50")` |
| **Model string** | `LocalVisualTCAV(model="resnet50", ...)` |
| **Model nn.Module** | `LocalVisualTCAV(model=my_resnet, model_name="resnet50", ...)` |
| **Configurable paths** | `concept_base_dir`, `random_dir`, `cache_dir` — all free |
| **Skip predict()** | `explain()` calls it automatically |
| **Use cache (default)** | `tcav.explain()` |
| **Force recompute** | `tcav.explain(force_recompute=True)` |
| **Clear cache** | `tcav.clear_cache()` |
| **Disable cache** | `LocalVisualTCAV(..., cache_dir=None, ...)` |
| **Multiple layers** | `layer_names=["layer2", "layer3", "layer4"]` |
| **Custom CAV** | `LocalVisualTCAV(cav_fn=my_function, ...)` |
| **Global explanation** | `GlobalVisualTCAV(test_images_dir=..., ...)` |
| **Save figures** | `plot(save_path="./output.png")` |
| **CLI** | `visual-tcav local --model resnet50 --image ./zebra.jpg ...` |

---

**GitHub:** https://github.com/saracavallini01/visual-tcav  
**PyPI:** `pip install visual-tcav`